# Can we predict next-month arrears?
Target: `arrears_next_month`. Data: one row per account per billing month.

Key design decisions:
- **`final_notice_sent` is excluded.** Per the data dictionary it is recorded in the billing cycle *after* the billing month, so it isn't known at prediction time (and is likely a consequence of arrears). We quantify the leakage below.
- **Time-based split.** Train on earlier months, test on the latest months, as in deployment. Accounts repeat across months, so a random split would leak account identity.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['billing_month']).sort_values('billing_month').reset_index(drop=True)
print(df.shape, df.isna().sum().sum(), 'missing;', df.duplicated(['meter_ref', 'billing_month']).sum(), 'duplicate account-months')
print(df.meter_ref.nunique(), 'accounts;', df.billing_month.min().date(), 'to', df.billing_month.max().date())
df.describe(include='all').T

## Target and leakage check

In [ ]:
print('Base rate:', df.arrears_next_month.mean().round(4))
print(df.groupby('final_notice_sent').arrears_next_month.agg(['mean', 'size']))
print(df.groupby('late_payments_12m').arrears_next_month.agg(['mean', 'size']))
print(df.groupby('tariff').arrears_next_month.agg(['mean', 'size']))
df.groupby('billing_month').arrears_next_month.mean().plot(title='Arrears rate by billing month'); plt.show()

`final_notice_sent` almost determines the outcome (~87% vs ~1.4% arrears), which is what you'd expect from a post-outcome field. It is dropped from the main models.

## Features and split

In [ ]:
df['bill_per_kwh'] = df.bill_eur / df.kwh
df['kwh_per_occupant'] = df.kwh / df.occupants
df['month'] = df.billing_month.dt.month
cat = ['tariff']
num = ['occupants', 'kwh', 'bill_eur', 'late_payments_12m', 'bill_per_kwh', 'kwh_per_occupant', 'month']
cutoff = df.billing_month.sort_values().unique()[-6]   # last 6 months = test
tr, te = df[df.billing_month < cutoff], df[df.billing_month >= cutoff]
print('train', len(tr), tr.arrears_next_month.mean().round(4), '| test', len(te), te.arrears_next_month.mean().round(4), '| cutoff', pd.Timestamp(cutoff).date())
print('accounts in both:', len(set(tr.meter_ref) & set(te.meter_ref)), 'of', te.meter_ref.nunique())

## Models

In [ ]:
def models(num, cat):
    return {'logistic': make_pipeline(ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(drop='first'), cat)]),
                                      LogisticRegression(max_iter=1000)),
            'grad_boost': make_pipeline(ColumnTransformer([('n', 'passthrough', num), ('c', OneHotEncoder(drop='first'), cat)]),
                                        HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))}
def evaluate(num, cat):
    out, fit = {}, {}
    for k, m in models(num, cat).items():
        m.fit(tr[num + cat], tr.arrears_next_month)
        p = m.predict_proba(te[num + cat])[:, 1]
        out[k] = (roc_auc_score(te.arrears_next_month, p), average_precision_score(te.arrears_next_month, p))
        fit[k] = (m, p)
    return pd.DataFrame(out, index=['ROC AUC', 'PR AUC']).T, fit
base = pd.DataFrame({'ROC AUC': [.5], 'PR AUC': [te.arrears_next_month.mean()]}, index=['no-skill'])
res, fit = evaluate(num, cat)
res1, _ = evaluate(num + ['final_notice_sent'], cat)
res1.index = [i + ' + final_notice (leaky)' for i in res1.index]
pd.concat([base, res, res1]).round(3)

The leaky variants look much better, but that information wouldn't be available when making the prediction. The honest numbers are the first block.

In [ ]:
y = te.arrears_next_month
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, (m, p) in fit.items():
    f, t, _ = roc_curve(y, p); ax[0].plot(f, t, label=k)
    pr, rc, _ = precision_recall_curve(y, p); ax[1].plot(rc, pr, label=k)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(title='ROC (test)', xlabel='FPR', ylabel='TPR')
ax[1].axhline(y.mean(), color='k', ls=':'); ax[1].set(title='Precision-recall (test)', xlabel='Recall', ylabel='Precision')
ax[0].legend(); plt.show()
p = fit['grad_boost'][1]
d = pd.DataFrame({'y': y.values, 'p': p}); d['decile'] = pd.qcut(d.p.rank(method='first'), 10, labels=range(1, 11))
d.groupby('decile', observed=True).y.mean().plot.bar(title='Actual arrears rate by predicted-risk decile (10 = highest)'); plt.show()

## What drives the prediction?

In [ ]:
m = fit['grad_boost'][0]
pi = permutation_importance(m, te[num + cat], y, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=num + cat).sort_values().plot.barh(title='Permutation importance (drop in test AUC)'); plt.show()

## Calibration under drift
The arrears rate climbs from ~2% (2023-24) to ~26% (Dec 2025) while the average `late_payments_12m` stays flat, so something outside these features changed. Ranking can survive that; absolute probabilities will not.

In [ ]:
pl = fit['logistic'][1]
print('test actual rate: %.3f | mean predicted (logistic): %.3f | (grad_boost): %.3f' % (y.mean(), pl.mean(), fit['grad_boost'][1].mean()))
# precision when flagging the top 10% / 20% riskiest accounts
for q in (0.1, 0.2):
    thr = np.quantile(p, 1 - q); sel = p >= thr
    print('top %d%%: precision %.2f, recall %.2f' % (q * 100, y[sel].mean(), y[sel].sum() / y.sum()))

## Conclusion
Yes, partially. Using only information available at billing time (tariff, occupants, usage, bill, late-payment history) the models rank accounts well above chance on the held-out latest 6 months (ROC AUC ~0.73, PR AUC ~0.50 vs a 0.22 no-skill baseline). `late_payments_12m` is the main signal.

Caveats:
- **`final_notice_sent` is leakage** (AUC ~0.97 with it). It is recorded after the billing month, so it was excluded. If it is in fact known at prediction time, tell me, because the question changes.
- **The arrears rate is drifting upward** (~2% to ~26%), driven by something not in the features. Use the model for risk *ranking*; predicted probabilities are not calibrated to the current rate (see the calibration cell) and would need recalibrating on recent data.
- One test window, no hyperparameter tuning, and no per-account history features (previous months' bills, trends), which could add signal.